In [29]:
import os
import sqlite3
from typing import List, Optional
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from tools import list_files, read_file
from context import SYSTEM_PROMPT
from dotenv import load_dotenv
load_dotenv()



True

In [30]:
DB_URL = "sqlite:///chat_history.sqlite"  # Automatically created locally
# GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


available_tools = [list_files, read_file]


In [31]:
# ------------------------------------------------------------------
# 2. Define Request/Response Models
# ------------------------------------------------------------------
class ChatRequest(BaseModel):
    session_id: str
    message: str

class ChatResponse(BaseModel):
    session_id: str
    response: str
    tools_used: str
    input_tokens: str
    output_tokens: str
    total_tokens: str

class ChatMessageItem(BaseModel):
    type: str  # e.g., 'human', 'ai', 'system', 'tool'
    content: str

class ChatHistoryResponse(BaseModel):
    session_id: str
    messages: List[ChatMessageItem]

class AgenticResponse(BaseModel):
    message: str = Field(description="A brief response to user query")
    tools_used: List[str] = Field(description="List of tools used for answering user queries")


In [32]:
# 1. Initialize the SQLite connection
# check_same_thread=False is safe because SqliteSaver uses internal locking
conn = sqlite3.connect("chat_history.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

# ------------------------------------------------------------------
# 3. LangChain Agent Setup
# ------------------------------------------------------------------
# llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.0)
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0.0)
agent = create_agent(
    model=llm,
    tools=available_tools,
    checkpointer=checkpointer,
    system_prompt=SYSTEM_PROMPT,
    response_format=AgenticResponse  # <--- Forces agent to produce structured output
)


In [33]:
session_id = "alskdlksdks"
message = "How many files are locally present ?"

In [34]:
config = { "configurable": { "thread_id": session_id }}
# Run the agent with context
response = agent.invoke({
    "messages": [
        {
            "role": "user", "content": message
        }
    ]
},
config=config,
version="v2"
)

agent_response: AgenticResponse = response["structured_response"]


/tmp/ipykernel_18804/4039474039.py:14: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  agent_response: AgenticResponse = response["structured_response"]


In [43]:
response["messages"][-1].response_metadata["token_usage"]

print(response["messages"][-1].response_metadata["token_usage"]["completion_tokens"])
print(response["messages"][-1].response_metadata["token_usage"]["prompt_tokens"])
print(response["messages"][-1].response_metadata["token_usage"]["total_tokens"])

31
447
478


/tmp/ipykernel_18804/754581994.py:1: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  response["messages"][-1].response_metadata["token_usage"]
/tmp/ipykernel_18804/754581994.py:3: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(response["messages"][-1].response_metadata["token_usage"]["completion_tokens"])
/tmp/ipykernel_18804/754581994.py:4: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(response["messages"][-1].response_metada

In [44]:
state = agent.get_state(config)

Deserializing unregistered type __main__.AgenticResponse from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'AgenticResponse')]


In [45]:
msgs = state.values.get("messages", [])

In [47]:
user_prompts = [msg.content for msg in msgs if msg.type == "human"]

In [48]:
user_prompts = "".join(user_prompts)

In [49]:
len(user_prompts)

36